In [0]:
import pandas as pd 
import os
import re
import pyspark
import datetime
import logging
import json
import requests
import urllib3
import time
from datetime import datetime
from pathlib import Path
from pyspark.sql import SparkSession
import configparser
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, BooleanType, ArrayType
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.ml import Pipeline
from pyspark.ml.feature import RegexTokenizer, NGram, CountVectorizer, IDF
from pyspark.ml.linalg import SparseVector, DenseVector
from pyspark.sql.types import DoubleType

def cluster_documents_from_similarity(
    catalog="dataplatform01_central_dev_catalog_01",
    schema="silver_sap_bo",
    sim_table="webi_metadata_cms_doc_global_similarity",
    out_table="webi_metadata_cms_doc_clusters",
    min_cosine=0.88,           # similarity threshold to create edges
    require_mutual=True,       # only keep edges if both directions exist
    algorithm="connected",     # "connected" or "union_find"
    zorder=True,
    logger=None
):
    """
    Create document clusters from the global similarity table using a graph approach.
    - Reads {catalog}.{schema}.{sim_table} with columns: src_document_id, nbr_document_id, cosine_similarity
    - Builds undirected edges for pairs with cosine >= min_cosine (optionally mutual-only)
    - Computes clusters via Connected Components (GraphFrames) or a union-find fallback
    - Writes {catalog}.{schema}.{out_table} with: Document_Id, cluster_id, cluster_size
    """
    from pyspark.sql import functions as F

    spark.sql(f"USE CATALOG {catalog}")
    spark.sql(f"USE SCHEMA {schema}")

    # 1) Load similarity pairs & filter by threshold
    df = spark.table(f"{catalog}.{schema}.{sim_table}") \
              .select("src_document_id", "nbr_document_id", "cosine_similarity") \
              .where(F.col("cosine_similarity") >= F.lit(min_cosine))

    # 2) Optional mutual edges (A↔B must both exist)
    if require_mutual:
        p1 = df.alias("p1")
        p2 = df.alias("p2")
        mutual = p1.join(
            p2,
            (F.col("p1.src_document_id") == F.col("p2.nbr_document_id")) &
            (F.col("p1.nbr_document_id") == F.col("p2.src_document_id")),
            "inner"
        )
        # Use the weaker of the two similarities as edge weight
        edges_directed = mutual.select(
            F.col("p1.src_document_id").alias("src"),
            F.col("p1.nbr_document_id").alias("dst"),
            F.least(F.col("p1.cosine_similarity"), F.col("p2.cosine_similarity")).alias("cosine_similarity")
        ).dropDuplicates(["src", "dst"])
    else:
        edges_directed = df.select(
            F.col("src_document_id").alias("src"),
            F.col("nbr_document_id").alias("dst"),
            F.col("cosine_similarity")
        ).dropDuplicates(["src", "dst"])

    # 3) Make edges undirected (add reversed edges) and dedupe
    edges_rev = edges_directed.select(
        F.col("dst").alias("src"),
        F.col("src").alias("dst"),
        F.col("cosine_similarity")
    )
    edges_und = edges_directed.unionByName(edges_rev).dropDuplicates(["src", "dst"])
    logger.info("Finished prep edges")
    # 4) Build vertices
    vertices = edges_und.select(F.col("src").alias("id")).union(
        edges_und.select(F.col("dst").alias("id"))
    ).distinct()

    # If there are no edges (all singletons), guard early
    if vertices.head(1) == []:
        # Create empty clusters table
        spark.createDataFrame([], "Document_Id STRING, cluster_id STRING, cluster_size INT") \
             .write.mode("overwrite").format("delta").saveAsTable(f"{catalog}.{schema}.{out_table}")
        return f"{catalog}.{schema}.{out_table}"
    logger.info("Finished building edges")
    # 5) Cluster: try GraphFrames connected components; fallback to union-find
    clusters_df = None
    if algorithm == "connected":
        try:
            from graphframes import GraphFrame
            g = GraphFrame(vertices, edges_und)
            cc = g.connectedComponents()
            clusters_df = cc.select(
                F.col("id").alias("Document_Id"),
                F.col("component").cast("string").alias("cluster_id")
            )
        except Exception as e:
            print(f"GraphFrames not available ({e}); falling back to union-find.")
            algorithm = "union_find"

    if algorithm == "union_find":
        # Driver-side union-find fallback (OK for moderate sizes)
        pairs = edges_und.select("src", "dst").distinct().collect()
        ids = vertices.select("id").distinct().collect()
        ids_list = [r["id"] for r in ids]

        parent = {i: i for i in ids_list}

        def find(x):
            while parent[x] != x:
                parent[x] = parent[parent[x]]
                x = parent[x]
            return x

        def union(a, b):
            ra, rb = find(a), find(b)
            if ra != rb:
                parent[rb] = ra

        for r in pairs:
            union(r["src"], r["dst"])

        rows = [(i, find(i)) for i in ids_list]
        clusters_df = spark.createDataFrame(rows, ["Document_Id", "cluster_id"])
    logger.info("Finished: Cluster: try GraphFrames connected components; fallback to union-find")
    # 6) Metrics: cluster size & persist
    cluster_sizes = clusters_df.groupBy("cluster_id").agg(F.count("*").alias("cluster_size"))
    out = clusters_df.join(cluster_sizes, on="cluster_id", how="left")

    out.write.mode("overwrite").format("delta").saveAsTable(f"{catalog}.{schema}.{out_table}")

    if zorder:
        try:
            spark.sql(f"OPTIMIZE {catalog}.{schema}.{out_table} ZORDER BY (cluster_id)")
        except Exception as e:
            print(f"OPTIMIZE skipped: {e}")

    logger.info(f"Clusters written to {catalog}.{schema}.{out_table}")

spark = SparkSession.builder \
    .appName("Clustering SQL") \
    .getOrCreate()
Todyay_Date = datetime.now()
local_directory=os.getcwd()
logs_path=os.path.join(local_directory,f"{local_directory}/logs")
Path(logs_path).mkdir(parents=True, exist_ok=True)
logs_path=os.path.join(logs_path, f"clustering_script_{Todyay_Date}.log")

logger = logging.getLogger("clustering_notebook_logger ")
logger.setLevel(logging.INFO)

if logger.handlers:
    for h in list(logger.handlers):
        logger.removeHandler(h)

fh = logging.FileHandler(logs_path, mode="a", encoding="utf-8")

fmt = logging.Formatter("%(asctime)s | %(levelname)s | %(message)s")
fh.setFormatter(fmt)
logger.addHandler(fh)

logger.info("Clustering Notebook started.")

cluster_table = cluster_documents_from_similarity(
    catalog="dataplatform01_central_dev_catalog_01",
    schema="silver_sap_bo",
    sim_table="webi_metadata_cms_doc_global_similarity_v25",
    out_table="webi_metadata_cms_doc_clusters_v25",
    min_cosine=0.7,
    require_mutual=True,
    algorithm="connected",
    logger=logger
)
cluster_table